In [3]:
import cv2
import numpy as np
from pathlib import Path
from boxmot.trackers import BotSort
from ultralytics import YOLO

In [4]:
YOLO_MODEL_PATH = "/mnt/d/Computer-Vision/Projects/Project2/runs/detect/train-3/weights/best.pt"

In [5]:
model = YOLO(YOLO_MODEL_PATH)

In [6]:
tracker = BotSort(
    reid_weights=Path('osnet_x1_0_msmt17.pt'),
    device='0',
    half=False,
    with_reid=True,
)

INFO     BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,          
         asso_func=iou, reid_model=None, track_high_thresh=0.5, track_low_thresh=0.1, new_track_thresh=0.6,        
         track_buffer=30, match_thresh=0.8, proximity_thresh=0.5, appearance_thresh=0.25, cmc_method=ecc,          
         frame_rate=30, fuse_first_associate=False, with_reid=True

In [7]:
VIDEO_PATH = "video.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)

In [8]:
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

In [9]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, stream=True)

    for result in results:
        boxes = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        labels = result.boxes.cls.cpu().numpy()

        detections = np.hstack((boxes, scores[:, np.newaxis], labels[:, np.newaxis]))

        if detections.size > 0:
            tracks = tracker.update(detections, frame)

            if tracks.size > 0:
                tracker.plot_results(frame, show_trajectories=False)



    cv2.imshow("YOLO + BoT-SORT Tracking", frame)
    if cv2.waitKey(1) == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()